### Load sample files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [21]:
!git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
%cd RAG-with-Python-Cookbook
!git sparse-checkout init --cone
!git sparse-checkout set datasets
!git checkout
!cp -r datasets /content/datasets


Cloning into 'RAG-with-Python-Cookbook'...
remote: Enumerating objects: 593, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 593 (delta 88), reused 29 (delta 15), pack-reused 450 (from 2)
Receiving objects: 100% (593/593), 39.74 MiB | 18.34 MiB/s, done.
Resolving deltas: 100% (252/252), done.
/content/RAG-with-Python-Cookbook/RAG-with-Python-Cookbook/RAG-with-Python-Cookbook/RAG-with-Python-Cookbook
Your branch is up to date with 'origin/main'.


### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [16]:
from google.colab import userdata
import os

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in Colab Secrets")

os.environ["OPENAI_API_KEY"] = api_key

## Prerequisits

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to uncomment and run the following codeblock to install the dependencies for this chapter.

In [17]:
!pip install chromadb

In [24]:
import chromadb
import openai

# tag::chunk_text[]
def chunk_text(text, chunk_size, overlap):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size

        if end < len(text):
            break_point = text.rfind("\n\n", start, end)
            if break_point == -1:
                break_point = text.rfind(". ", start, end)
            if break_point != -1 and break_point > start:
                end = break_point + 1

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        start = end - overlap if end < len(text) else end

    return chunks


# end::chunk_text[]


# tag::generate_embeddings[]
def generate_embeddings(texts, client, model):
    embeddings = []
    batch_size = 100

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        response = client.embeddings.create(model=model, input=batch)
        embeddings.extend([item.embedding for item in response.data])

    return embeddings


# end::generate_embeddings[]


# tag::ingest_to_chromadb[]
def ingest_to_chromadb(chunks, embeddings, db_path, collection_name):

    db_path.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(db_path))

    try:
        client.delete_collection(name=collection_name)
    except:
        pass

    collection = client.create_collection(
        name=collection_name, metadata={"description": "Harry Potter knowledge base"}
    )

    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        embeddings=embeddings,
        documents=chunks,
        metadatas=[{"chunk_index": i} for i in range(len(chunks))],
    )

    return collection.count()


# end::ingest_to_chromadb[]


# tag::run_ingestion[]
from pathlib import Path
from openai import OpenAI
import os

KNOWLEDGE_BASE_FILE = "../datasets/text_files/harry_potter.txt"  # Path to your knowledge base file
CHUNK_SIZE = 1000  # Number of characters per chunk
CHUNK_OVERLAP = 200  # Number of overlapping characters between chunks
EMBEDDING_MODEL = "text-embedding-ada-002"  # OpenAI embedding model name
CHROMA_DB_DIR = Path("chroma_db")  # Directory for ChromaDB persistence
COLLECTION_NAME = "harry_potter_kb"  # Name of the ChromaDB collection

with open(KNOWLEDGE_BASE_FILE, "r", encoding="utf-8") as f:
    text = f.read()

chunks = chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)

client = OpenAI()
embeddings = generate_embeddings(chunks, client, EMBEDDING_MODEL)

count = ingest_to_chromadb(chunks, embeddings, CHROMA_DB_DIR, COLLECTION_NAME)
# end::run_ingestion[]


FileNotFoundError: [Errno 2] No such file or directory: '../datasets/text_files/harry_potter.txt'